In [1]:
import pandas as pd
import numpy as np

# =====================================
# 1. LOAD DATA
# =====================================
def load_data(file_path):
    df = pd.read_csv(file_path)
    return df


# =====================================
# 2. CLEAN SENSOR DATA (IMPORTANT FIX)
# =====================================
def get_sensor_data(df, time_col=None):
    """
    Removes ID column + keeps only real sensor signals
    Removes duplicate columns (critical for HD)
    """

    # Remove ID / time column
    if time_col and time_col in df.columns:
        df = df.drop(columns=[time_col])

    # Keep only numeric sensor columns
    sensor_df = df.select_dtypes(include=[np.number])

    # Remove duplicate columns (VERY IMPORTANT for your dataset)
    sensor_df = sensor_df.loc[:, ~sensor_df.T.duplicated()]

    return sensor_df


# =====================================
# 3. CREATE ROLLING WINDOWS
# =====================================
def create_windows(sensor_df, window_size, step_size):
    windows = []

    for start in range(0, len(sensor_df) - window_size + 1, step_size):
        end = start + window_size

        window = sensor_df.iloc[start:end].reset_index(drop=True)
        windows.append(window)

    return windows


# =====================================
# 4. STRUCTURE WINDOWS (FOR ML / ANALYSIS)
# =====================================
def structure_windows(windows):
    return np.array([w.values for w in windows])


# =====================================
# 5. CORRELATION PER WINDOW
# =====================================
def compute_correlations(windows):
    results = []

    for i, w in enumerate(windows):
        corr = w.corr()

        results.append({
            "window_id": i,
            "correlation_matrix": corr
        })

    return results


# =====================================
# 6. FULL PIPELINE
# =====================================
def rolling_window_pipeline(file_path,
                            window_size=50,
                            step_size=25,
                            time_col="entry_id"):

    # Load data
    df = load_data(file_path)

    # Clean sensors (FIXED)
    sensor_df = get_sensor_data(df, time_col)

    # Create windows
    windows = create_windows(sensor_df, window_size, step_size)

    # Structure
    structured_data = structure_windows(windows)

    # Correlation
    correlations = compute_correlations(windows)

    return windows, structured_data, correlations


# =====================================
# 7. RUN
# =====================================
if __name__ == "__main__":

    file_path = "2881821.csv"

    windows, structured_data, correlations = rolling_window_pipeline(
        file_path,
        window_size=50,
        step_size=25,
        time_col="entry_id"   
    )

    print("Number of windows:", len(windows))
    print("Structured shape:", structured_data.shape)

    print("\nFirst window:")
    print(windows[0])

    print("\nCorrelation (Window 0):")
    print(correlations[0]["correlation_matrix"])

Number of windows: 30
Structured shape: (30, 50, 5)

First window:
    field1  field2  field3  field4  field5
0      6.0     6.0     3.6     5.0     0.0
1      6.0     4.0     2.4     3.0    10.0
2      6.0     7.0     4.2     6.0     3.0
3      9.0     4.0     3.6     5.0    13.0
4      4.0     4.0     1.6     3.0    22.0
5      6.0     5.0     3.0     8.0    31.0
6      3.0     4.0     1.2     7.0    41.0
7      4.0     5.0     2.0     3.0    51.0
8      7.0     7.0     4.9     5.0    61.0
9      6.0     6.0     3.6     6.0    71.0
10     5.0     7.0     3.5     5.0    81.0
11     7.0     4.0     2.8     5.0    91.0
12     9.0     4.0     3.6     5.0   101.0
13     6.0     5.0     3.0     3.0   111.0
14     8.0     6.0     4.8     4.0   121.0
15     8.0     4.0     3.2     4.0   131.0
16     7.0     3.0     2.1     5.0   141.0
17     5.0    10.0     5.0     6.0   151.0
18     5.0     3.0     1.5     3.0   161.0
19     6.0     8.0     4.8     4.0   171.0
20     2.0    12.0     2.4    